# Optional agent alignment and prompt optimization lab

This connected, disabled-by-default lab adapts MLflow's [agent alignment and optimization cookbook](https://mlflow.org/cookbook/agent-alignment-optimization/) to the repository's stricter release lifecycle.

The safe sequence is:

1. calibrate a domain judge against human feedback;
2. validate the frozen judge on held-out labels;
3. optimize an exact seed prompt against a separate training split and bounded request budget;
4. register the optimized text as a new immutable prompt version;
5. evaluate that exact version on a final held-out release split;
6. let the ordinary release gate decide `adopt`, `reject`, or `inconclusive`.

This notebook never moves `production`. Optimization proposes a change; it does not authorize deployment.

## 1. Lock three disjoint evidence splits

Judge calibration, optimizer training, and final release testing must not reuse cases. Otherwise the aligned judge and optimized prompt can certify the same examples they learned from.

In [ ]:
import hashlib
import json

SPLIT_MANIFEST = {
    "judge_calibration": [
        "cal-revenue-01",
        "cal-margin-01",
        "cal-guidance-01",
        "cal-cash-01",
        "cal-risk-01",
        "cal-policy-01",
    ],
    "optimizer_training": [
        "train-revenue-01",
        "train-margin-01",
        "train-guidance-01",
        "train-cash-01",
        "train-risk-01",
        "train-policy-01",
    ],
    "held_out_release": [
        "holdout-revenue-01",
        "holdout-margin-01",
        "holdout-guidance-01",
        "holdout-cash-01",
        "holdout-risk-01",
        "holdout-policy-01",
    ],
}

split_sets = {name: set(case_ids) for name, case_ids in SPLIT_MANIFEST.items()}
assert split_sets["judge_calibration"].isdisjoint(split_sets["optimizer_training"])
assert split_sets["judge_calibration"].isdisjoint(split_sets["held_out_release"])
assert split_sets["optimizer_training"].isdisjoint(split_sets["held_out_release"])
assert sum(len(case_ids) for case_ids in split_sets.values()) == len(
    set().union(*split_sets.values())
)

SPLIT_MANIFEST_DIGEST = hashlib.sha256(
    json.dumps(
        SPLIT_MANIFEST,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()
{
    "split_manifest_digest": SPLIT_MANIFEST_DIGEST,
    "case_counts": {name: len(case_ids) for name, case_ids in split_sets.items()},
}

## 2. Make experimental dependencies and spend bounds explicit

`MemAlignOptimizer` is experimental and requires DSPy. `GepaPromptOptimizer` requires GEPA. Neither dependency is present in this repository's certified locks, so the connected lab must remain off unless dependency policy, exact locks, template locks, and compatibility evidence are updated together.

In [ ]:
import importlib.util

OPTIMIZATION_BUDGET = {
    "max_metric_calls": 30,
    "max_training_cases": len(SPLIT_MANIFEST["optimizer_training"]),
    "evaluation_concurrency": 1,
    "request_timeout_seconds": 60,
}
EXPERIMENTAL_DEPENDENCIES = {
    "dspy": importlib.util.find_spec("dspy") is not None,
    "gepa": importlib.util.find_spec("gepa") is not None,
}
{
    "optimization_budget": OPTIMIZATION_BUDGET,
    "experimental_dependencies": EXPERIMENTAL_DEPENDENCIES,
    "ready": all(EXPERIMENTAL_DEPENDENCIES.values()),
}

## 3. Connected alignment and optimization skeleton

Before enabling the next cell:

- collect at least the approved number of balanced human labels with rationales and source `group:domain-reviewers`;
- use the same assessment name for human feedback and the judge;
- freeze and validate the aligned judge before optimization;
- set the exact seed prompt, optimizer data, governed reflection-model URI, and validated judge version/run evidence;
- make `predict_fn` load and format the registered seed prompt **inside every call**. Building an agent once around the baseline text would make optimizer candidates inert.

In [ ]:
RUN_EXPERIMENTAL_OPTIMIZATION = False
JUDGE_NAME = "uncertainty_explanation"
JUDGE_EXPERIMENT_ID = None
SEED_PROMPT_URI = None  # Exact URI: prompts:/<qualified-name>/<version>
REFLECTION_MODEL_URI = None  # Resolve a governed logical model keylessly.
ALIGNED_JUDGE_VERSION = None
JUDGE_VALIDATION_RUN_ID = None
JUDGE_VALIDATION_AGREEMENT = None
JUDGE_VALIDATION_LABEL_COUNT = None
OPTIMIZER_TRAIN_DATA = None

if RUN_EXPERIMENTAL_OPTIMIZATION:
    required_values = {
        "JUDGE_EXPERIMENT_ID": JUDGE_EXPERIMENT_ID,
        "SEED_PROMPT_URI": SEED_PROMPT_URI,
        "REFLECTION_MODEL_URI": REFLECTION_MODEL_URI,
        "ALIGNED_JUDGE_VERSION": ALIGNED_JUDGE_VERSION,
        "JUDGE_VALIDATION_RUN_ID": JUDGE_VALIDATION_RUN_ID,
        "JUDGE_VALIDATION_AGREEMENT": JUDGE_VALIDATION_AGREEMENT,
        "JUDGE_VALIDATION_LABEL_COUNT": JUDGE_VALIDATION_LABEL_COUNT,
        "OPTIMIZER_TRAIN_DATA": OPTIMIZER_TRAIN_DATA,
    }
    missing_values = [
        name
        for name, value in required_values.items()
        if value is None or (isinstance(value, str) and not value.strip())
    ]
    if missing_values:
        raise ValueError(f"Configure the connected lab first: {missing_values}")
    if not all(EXPERIMENTAL_DEPENDENCIES.values()):
        raise RuntimeError(
            "DSPy and GEPA are not in the certified locks; complete the dependency "
            "policy and compatibility workflow before enabling this lab"
        )
    if JUDGE_VALIDATION_AGREEMENT < 0.75 or JUDGE_VALIDATION_LABEL_COUNT < 50:
        raise RuntimeError(
            "The aligned judge has not passed held-out agreement and sample-size "
            "requirements"
        )

    import mlflow
    from mlflow.genai.optimize import GepaPromptOptimizer
    from mlflow.genai.scorers import get_scorer

    from examples.notebook_setup import (
        preflight_databricks,
        prepare_notebook_environment,
    )

    environment = prepare_notebook_environment(
        evidence_destination="databricks"
    )
    connected = preflight_databricks(environment)
    model = connected.context.providers.model("general-chat")
    aligned_judge = get_scorer(
        name=JUDGE_NAME,
        experiment_id=JUDGE_EXPERIMENT_ID,
        version=int(ALIGNED_JUDGE_VERSION),
    )
    print(
        {
            "aligned_judge": aligned_judge.name,
            "aligned_judge_version": ALIGNED_JUDGE_VERSION,
            "validation_run_id": JUDGE_VALIDATION_RUN_ID,
            "validation_agreement": JUDGE_VALIDATION_AGREEMENT,
            "validation_label_count": JUDGE_VALIDATION_LABEL_COUNT,
        }
    )

    def predict_with_registered_prompt(
        question,
        earnings_excerpt,
        source_id,
    ):
        active_prompt = mlflow.genai.load_prompt(SEED_PROMPT_URI)
        rendered = active_prompt.format(
            question=question,
            earnings_excerpt=earnings_excerpt,
            source_id=source_id,
        )
        response = model.generate(
            [{"role": "system", "content": rendered}],
            temperature=0.0,
            max_tokens=400,
        )
        return response.content

    optimization_result = mlflow.genai.optimize_prompts(
        predict_fn=predict_with_registered_prompt,
        train_data=OPTIMIZER_TRAIN_DATA,
        prompt_uris=[SEED_PROMPT_URI],
        optimizer=GepaPromptOptimizer(
            reflection_model=REFLECTION_MODEL_URI,
            max_metric_calls=OPTIMIZATION_BUDGET["max_metric_calls"],
            display_progress_bar=True,
        ),
        scorers=[aligned_judge],
    )
    print(
        {
            "initial_score": optimization_result.initial_eval_score,
            "final_score": optimization_result.final_eval_score,
            "split_manifest_digest": SPLIT_MANIFEST_DIGEST,
            "budget": OPTIMIZATION_BUDGET,
            "next": "register a new immutable version; do not move an alias",
        }
    )
else:
    print("EXPERIMENTAL ALIGNMENT AND OPTIMIZATION SKIPPED")

## 4. Register, test held-out cases, then use the normal gate

If optimization produces a useful template:

1. register it as a new immutable version of the same qualified prompt;
2. record its content digest, exact URI, seed URI, split-manifest digest, optimizer/reflection model, dependency digest, request budget, and source commit;
3. load that exact new version inside every prediction on `held_out_release`;
4. run deterministic fact, citation, policy, critical-row, latency, token, cost, and cost-coverage gates, with the validated judge as additional evidence;
5. compare against the untouched baseline and choose `adopt`, `reject`, or `inconclusive`;
6. move a controlled alias only in the ordinary release workflow after all checks pass.

Do not inspect private judge fields such as `_semantic_memory`; record public versioned instructions and alignment evidence.

In [ ]:
{
    "stage": "optimization_plan",
    "split_manifest_digest": SPLIT_MANIFEST_DIGEST,
    "experimental_dependencies_ready": all(EXPERIMENTAL_DEPENDENCIES.values()),
    "decision": "inconclusive",
    "release": "blocked",
    "reason": (
        "optimization is disabled and cannot authorize release; register any "
        "proposed prompt and run the final held-out gate"
    ),
}